In [ ]:

import zipfile
import pandas as pd
import os
import numpy as np
from pathlib import Path
from config import DATA_ROOT_DIR,FIGURES_DIR,RESULTS_DIR, VARIABLES
from util import load_dhs_kids_data
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

loco = pd.read_csv(f"{RESULTS_DIR}/anemia_loco_results.csv")

comparisons = [
    ("tabpfn_auc", "lg_auc", "LogReg"),
    ("tabpfn_auc", "lgbm_auc", "LightGBM"),
    ("tabpfn_auc", "xgb_auc", "XGBoost"),
]

rows, pvals = [], []

for tab_col, other_col, name in comparisons:
    diff = loco[tab_col] - loco[other_col]
    stat, p = wilcoxon(loco[tab_col], loco[other_col])

    rows.append({
        "Comparison": f"TabPFN vs. {name}",
        "Wins": (diff > 0).sum(),
        "N": len(diff),
        "Median $\\Delta$AUC": diff.median(),
        "Mean $\\Delta$AUC": diff.mean(),
        "$W$": stat,
        "$p$": p,
    })
    pvals.append(p)

# Holm correction
_, p_adj, _, _ = multipletests(pvals, method="holm")

loco_table = pd.DataFrame(rows)
loco_table["Holm-adjusted $p$"] = p_adj

# Format numeric columns
latex_df = loco_table.copy()
latex_df["Median $\\Delta$AUC"] = latex_df["Median $\\Delta$AUC"].map("{:.3f}".format)
latex_df["Mean $\\Delta$AUC"] = latex_df["Mean $\\Delta$AUC"].map("{:.3f}".format)
latex_df["$W$"] = latex_df["$W$"].map("{:.1f}".format)
latex_df["$p$"] = latex_df["$p$"].map("{:.4f}".format)
latex_df["Holm-adjusted $p$"] = latex_df["Holm-adjusted $p$"].map("{:.4f}".format)

latex = latex_df.to_latex(
    index=False,
    escape=False,
    column_format="lccrrrr",
    caption=("Wilcoxon signed-rank tests comparing LOCO AUC--ROC performance "
             "of TabPFN against competing models across the 16 held-out countries. "
             "Holm-adjusted $p$-values account for multiple comparisons."),
    label="tab:loco_wilcoxon",
    bold_rows=False
)

print(latex)

# with open(f"{RESULTS_DIR}/loco_wilcoxon_table.tex", "w") as f:
#     f.write(latex)

In [ ]:
# Reverse-LOCO — country-level cluster bootstrap

# ------------------------------------------------------------------
# Load reverse-LOCO results
# ------------------------------------------------------------------
rev = pd.read_csv(f"{RESULTS_DIR}/anemia_reverse_loco_results.csv")
countries = rev["train_country"].unique()

n_boot = 5000
rng = np.random.default_rng(42)


# ------------------------------------------------------------------
# Country-level cluster bootstrap
# ------------------------------------------------------------------
def cluster_bootstrap_test(df, tab_col, other_col, countries, n_boot, rng):
    boot_diffs = []

    for _ in range(n_boot):
        # Resample countries (clusters) with replacement
        sampled = rng.choice(countries, size=len(countries), replace=True)

        # Include all directed pairs originating from sampled countries
        resampled_rows = pd.concat(
            [df[df["train_country"] == c] for c in sampled],
            ignore_index=True,
        )

        boot_diffs.append(
            (resampled_rows[tab_col] - resampled_rows[other_col]).mean()
        )

    boot_diffs = np.asarray(boot_diffs)

    ci_low, ci_high = np.percentile(boot_diffs, [2.5, 97.5])

    # Two-sided bootstrap p-value
    p_boot = 2 * min(
        (boot_diffs > 0).mean(),
        (boot_diffs < 0).mean()
    )

    return boot_diffs.mean(), ci_low, ci_high, p_boot


# ------------------------------------------------------------------
# Compute comparisons
# ------------------------------------------------------------------
results = []

comparisons = [
    ("tabpfn_auc", "lg_auc", "LogReg"),
    ("tabpfn_auc", "lgbm_auc", "LightGBM"),
    ("tabpfn_auc", "xgb_auc", "XGBoost"),
]

for tab_col, other_col, name in comparisons:

    mean_d, lo, hi, p = cluster_bootstrap_test(
        rev,
        tab_col,
        other_col,
        countries,
        n_boot,
        rng,
    )

    results.append({
        "Comparison": f"TabPFN vs. {name}",
        "Directed pairs": len(rev),
        "TabPFN wins (%)": 100 * (rev[tab_col] > rev[other_col]).mean(),
        r"Mean $\Delta$AUC": mean_d,
        r"95\% CI": f"({lo:.3f}, {hi:.3f})",
        r"$p$": p,
    })

rev_table = pd.DataFrame(results)

# ------------------------------------------------------------------
# Format for LaTeX
# ------------------------------------------------------------------
latex_df = rev_table.copy()

latex_df["TabPFN wins (%)"] = latex_df["TabPFN wins (%)"].map("{:.1f}".format)
latex_df[r"Mean $\Delta$AUC"] = latex_df[r"Mean $\Delta$AUC"].map("{:.3f}".format)
latex_df[r"$p$"] = latex_df[r"$p$"].map("{:.4f}".format)

latex = latex_df.to_latex(
    index=False,
    escape=False,
    column_format="lccccc",
    caption=(
        "Country-level cluster bootstrap comparison of reverse-LOCO "
        "AUC--ROC performance between TabPFN and competing models. "
        "The 95\\% confidence interval (CI) corresponds to the bootstrap "
        "distribution of the mean difference in AUC."
    ),
    label="tab:reverse_loco_cluster_bootstrap",
)

print(latex)

# Save LaTeX table
# with open(f"{RESULTS_DIR}/reverse_loco_cluster_bootstrap.tex", "w") as f:
#     f.write(latex)

In [ ]:
# Read results
sen_df = pd.read_csv(f"{RESULTS_DIR}/anemia_model_results_by_country_mice.csv")

# Clean country names
sen_df["country"] = (
    sen_df["country"]
    .str.replace("_", " ", regex=False)     # Replace underscores with spaces
    .str.replace("DHS", "", regex=False)    # Remove 'DHS'
    .str.replace(r"\s+", " ", regex=True)   # Remove any extra spaces
    .str.strip()                            # Trim leading/trailing spaces
)

# Model names
models = {
    "lg": "Logistic Regression",
    "lgbm": "LightGBM",
    "xgb": "XGBoost",
    "tabpfn": "TabPFN"
}

latex = []

latex.append(r"\begin{table}[ht]")
latex.append(r"\centering")
latex.append(r"\caption{AUC--ROC (95\% confidence intervals) for each model across DHS countries.}")
latex.append(r"\label{tab:auc_country}")
latex.append(r"\begin{tabular}{lcccc}")
latex.append(r"\toprule")
latex.append(r"Country & Logistic Regression & LightGBM & XGBoost & TabPFN \\")
latex.append(r"\midrule")

for _, row in sen_df.iterrows():

    vals = []

    for m in models.keys():
        auc = row[f"{m}_auc"]
        low = row[f"{m}_auc_ci_lower"]
        up = row[f"{m}_auc_ci_upper"]

        vals.append(f"{auc:.3f} ({low:.3f}--{up:.3f})")

    latex.append(
        f"{row['country']} & " +
        " & ".join(vals) +
        r" \\"
    )

latex.append(r"\bottomrule")
latex.append(r"\end{tabular}")
latex.append(r"\end{table}")

latex_table = "\n".join(latex)

# with open("auc_country_table.tex", "w") as f:
#     f.write(latex_table)

print(latex_table)

In [ ]:
sen_df  = pd.read_csv(f"{RESULTS_DIR}/anemia_model_results_by_country.csv")


# Clean country names
sen_df["country"] = (
    sen_df["country"]
    .str.replace("_", " ", regex=False)     # Replace underscores with spaces
    .str.replace("DHS", "", regex=False)    # Remove 'DHS'
    .str.replace(r"\s+", " ", regex=True)   # Remove any extra spaces
    .str.strip()                            # Trim leading/trailing spaces
)

latex = []

latex.append(r"\begin{table}[ht]")
latex.append(r"\centering")
latex.append(r"\caption{AUC--ROC (95\% confidence intervals) for each model across DHS countries.}")
latex.append(r"\label{tab:auc_country}")
latex.append(r"\begin{tabular}{lcccc}")
latex.append(r"\toprule")
latex.append(r"Country & Logistic Regression & LightGBM & XGBoost & TabPFN \\")
latex.append(r"\midrule")

for _, row in sen_df.iterrows():

    vals = []

    for m in models.keys():
        auc = row[f"{m}_auc"]
        low = row[f"{m}_auc_ci_lower"]
        up = row[f"{m}_auc_ci_upper"]

        vals.append(f"{auc:.3f} ({low:.3f}--{up:.3f})")

    latex.append(
        f"{row['country']} & " +
        " & ".join(vals) +
        r" \\"
    )

latex.append(r"\bottomrule")
latex.append(r"\end{tabular}")
latex.append(r"\end{table}")

latex_table = "\n".join(latex)

# with open("auc_country_table.tex", "w") as f:
#     f.write(latex_table)

print(latex_table)

In [ ]:
# Read results
sen_df = pd.read_csv(f"{RESULTS_DIR}/anemia_model_results_by_country_knn.csv")

# Clean country names
sen_df["country"] = (
    sen_df["country"]
    .str.replace("_", " ", regex=False)     # Replace underscores with spaces
    .str.replace("DHS", "", regex=False)    # Remove 'DHS'
    .str.replace(r"\s+", " ", regex=True)   # Remove any extra spaces
    .str.strip()                            # Trim leading/trailing spaces
)

# Model names
models = {
    "lg": "Logistic Regression",
    "lgbm": "LightGBM",
    "xgb": "XGBoost",
    "tabpfn": "TabPFN"
}

latex = []

latex.append(r"\begin{table}[ht]")
latex.append(r"\centering")
latex.append(r"\caption{AUC--ROC (95\% confidence intervals) for each model across DHS countries.}")
latex.append(r"\label{tab:auc_country}")
latex.append(r"\begin{tabular}{lcccc}")
latex.append(r"\toprule")
latex.append(r"Country & Logistic Regression & LightGBM & XGBoost & TabPFN \\")
latex.append(r"\midrule")

for _, row in sen_df.iterrows():

    vals = []

    for m in models.keys():
        auc = row[f"{m}_auc"]
        low = row[f"{m}_auc_ci_lower"]
        up = row[f"{m}_auc_ci_upper"]

        vals.append(f"{auc:.3f} ({low:.3f}--{up:.3f})")

    latex.append(
        f"{row['country']} & " +
        " & ".join(vals) +
        r" \\"
    )

latex.append(r"\bottomrule")
latex.append(r"\end{tabular}")
latex.append(r"\end{table}")

latex_table = "\n".join(latex)

# with open("auc_country_table.tex", "w") as f:
#     f.write(latex_table)

print(latex_table)

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

df = pd.read_csv(f"{RESULTS_DIR}/anemia_model_results_by_country.csv")

comparisons = [("tabpfn", "lg"), ("tabpfn", "lgbm"), ("tabpfn", "xgb")]
results = []

for model_a, model_b in comparisons:
    auc_a = df[f"{model_a}_auc"].values
    auc_b = df[f"{model_b}_auc"].values
    diff = auc_a - auc_b

    stat, p = wilcoxon(auc_a, auc_b)
    results.append({
        "comparison": f"TabPFN vs {model_b.upper()}",
        "n_wins_tabpfn": (auc_a > auc_b).sum(),
        "n_countries": len(auc_a),
        "median_delta_auc": np.median(diff),
        "mean_delta_auc": np.mean(diff),
        "wilcoxon_stat": stat,
        "p_value": p,
    })

results_df = pd.DataFrame(results)
results_df["p_adj_holm"] = multipletests(results_df["p_value"], method="holm")[1]

print(results_df)

In [ ]:
prev_df  = pd.read_csv(f"{RESULTS_DIR}/country_anemia_prevalence.csv")

prev_df

In [ ]:
missingness_df = pd.read_csv(f"{RESULTS_DIR}/country_predictor_missingness.csv")
missingness_df = missingness_df.rename(
    columns={"Variable": "Country"}
)
missingness_df


In [ ]:
# # ==============================================================
# # Generate predictor-level missingness tables for DHS datasets
# # ==============================================================

# import pandas as pd
# import numpy as np
# import os

# from config import VARIABLES, CAL_COLS, RESULTS_DIR
# from util import load_dhs_kids_data, preprocess_dhs


# # --------------------------------------------------------------
# # Load country metadata
# # --------------------------------------------------------------

# country_df = pd.read_csv("./dhs_country_file_summary.csv")


# TARGET   = VARIABLES[-1]
# FEATURES = VARIABLES[:-1]


# # --------------------------------------------------------------
# # DHS missing value codes
# # --------------------------------------------------------------

# ALL_MISSING = [
#     994, 995, 996, 997, 998, 999,
#     9998, 9999,
#     -1,
#     8, 9, 98, 99
# ]


# # --------------------------------------------------------------
# # Function: calculate missingness
# # --------------------------------------------------------------

# def calculate_missingness(df, variables, country):

#     results = []

#     analytic_n = len(df)

#     for var in variables:

#         if var not in df.columns:
#             continue

#         missing_count = df[var].isna().sum()

#         missing_pct = (
#             missing_count / analytic_n * 100
#             if analytic_n > 0 else np.nan
#         )

#         results.append(
#             {
#                 "Country": country,
#                 "Predictor": var,
#                 "Analytic sample size": analytic_n,
#                 "Missing count": missing_count,
#                 "Missing proportion (%)": round(missing_pct, 2)
#             }
#         )

#     return results



# # --------------------------------------------------------------
# # Main loop
# # --------------------------------------------------------------

# missingness_results = []


# for index, row in country_df.iterrows():

#     country = row["country"]

#     print("\n" + "=" * 70)
#     print(f"Processing {country}")
#     print("=" * 70)


#     # ----------------------------------------------------------
#     # Load DHS KR data
#     # ----------------------------------------------------------

#     data_dict = load_dhs_kids_data(
#         country_df,
#         indx=index,
#         variables=VARIABLES
#     )


#     if "kr" not in data_dict:

#         print("Skipping: no KR dataset")
#         continue


#     kr_df = data_dict["kr"]


#     # ----------------------------------------------------------
#     # Apply preprocessing
#     # ----------------------------------------------------------

#     df_clean = preprocess_dhs(
#         kr_df,
#         FEATURES,
#         CAL_COLS,
#         TARGET
#     )


#     print(
#         f"Final analytic sample: {df_clean.shape}"
#     )


#     # ----------------------------------------------------------
#     # Calculate predictor missingness
#     # ----------------------------------------------------------

#     country_missingness = calculate_missingness(
#         df_clean,
#         FEATURES,
#         country
#     )


#     missingness_results.extend(
#         country_missingness
#     )



# # --------------------------------------------------------------
# # Create full missingness dataframe
# # --------------------------------------------------------------

# missingness_df = pd.DataFrame(
#     missingness_results
# )


# # --------------------------------------------------------------
# # Save complete supplementary table
# # --------------------------------------------------------------

# full_output = (
#     f"{RESULTS_DIR}/country_predictor_missingness.csv"
# )


# missingness_df.to_csv(
#     full_output,
#     index=False
# )


# print("\nSaved full missingness table:")
# print(full_output)



# # ==============================================================
# # Generate manuscript summary table
# # ==============================================================


# missingness_summary = (
#     missingness_df
#     .groupby("Predictor")
#     .agg(
#         Mean_missingness_percent=(
#             "Missing proportion (%)",
#             "mean"
#         ),

#         Median_missingness_percent=(
#             "Missing proportion (%)",
#             "median"
#         ),

#         Maximum_missingness_percent=(
#             "Missing proportion (%)",
#             "max"
#         ),

#         Countries_with_missing_data=(
#             "Missing count",
#             lambda x: (x > 0).sum()
#         )
#     )
#     .reset_index()
# )


# missingness_summary[
#     [
#         "Mean_missingness_percent",
#         "Median_missingness_percent",
#         "Maximum_missingness_percent"
#     ]
# ] = missingness_summary[
#     [
#         "Mean_missingness_percent",
#         "Median_missingness_percent",
#         "Maximum_missingness_percent"
#     ]
# ].round(2)



# summary_output = (
#     f"{RESULTS_DIR}/predictor_missingness_summary.csv"
# )


# missingness_summary.to_csv(
#     summary_output,
#     index=False
# )


# print("\nSaved summary missingness table:")
# print(summary_output)



# # ==============================================================
# # Identify predictors with high missingness
# # ==============================================================

# high_missingness = (
#     missingness_df
#     .sort_values(
#         "Missing proportion (%)",
#         ascending=False
#     )
#     .head(20)
# )


# print("\nTop predictors by missingness:")
# print(high_missingness)



# # --------------------------------------------------------------
# # Preview
# # --------------------------------------------------------------

# print("\nFull missingness table:")
# print(missingness_df.head())


# print("\nSummary table:")
# print(missingness_summary.head())

In [ ]:
missingness_df

In [ ]:
high_missingness

In [ ]:
"""
compute_missingness_table.py

Generates the missingness table requested by the reviewer:
    "Include a table showing missingness proportions for all predictors."

Missingness is computed on the analytic sample AFTER preprocess_dhs()
(missing codes -> NaN, age filter applied, missing-target rows dropped)
and BEFORE prepare_ml_data() does any median/mode imputation. This is the
sample your models actually used, so the reported proportions match the
data that went into the analysis.

Outputs (written to RESULTS_DIR):
    missingness_by_country.csv   -- long format, one row per country x variable
    missingness_pooled.csv       -- one row per variable, pooled across all countries
    missingness_pooled.md        -- same pooled table as a markdown table (for manuscript / response letter)

Run:
    python compute_missingness_table.py
"""

import os
import pandas as pd
import numpy as np

from config import DATA_ROOT_DIR, RESULTS_DIR, VARIABLES, CAL_COLS
from util import load_dhs_kids_data, preprocess_dhs

# -----------------------------------------------------------------------------
# Setup
# -----------------------------------------------------------------------------
os.makedirs(RESULTS_DIR, exist_ok=True)

df = pd.read_csv("./dhs_country_file_summary.csv")

TARGET = VARIABLES[-1]
FEATURES = VARIABLES[:-1]

missingness_records = []          # per-country long-format rows
n_total = pd.Series(0, index=FEATURES, dtype=int)
n_missing_total = pd.Series(0, index=FEATURES, dtype=int)

failed_countries = []

# -----------------------------------------------------------------------------
# Loop over countries
# -----------------------------------------------------------------------------
for index, row in df.iterrows():

    country = row["country"]
    print(f"\n{'='*60}")
    print(f"Processing: {country}")
    print(f"{'='*60}")

    try:
        data_dict = load_dhs_kids_data(df, indx=index, variables=VARIABLES)

        df_clean = preprocess_dhs(
            data_dict["kr"],
            FEATURES,
            CAL_COLS,
            TARGET
        )
    except Exception as e:
        print(f"  WARNING: skipping {country} due to error: {e}")
        failed_countries.append(country)
        continue

    if len(df_clean) == 0:
        print(f"  WARNING: {country} has 0 eligible rows after preprocessing, skipping")
        failed_countries.append(country)
        continue

    # --- missingness snapshot on the analytic sample, pre-imputation ---
    present_features = [c for c in FEATURES if c in df_clean.columns]
    missing_features = [c for c in FEATURES if c not in df_clean.columns]
    if missing_features:
        print(f"  NOTE: features not found in df_clean and excluded: {missing_features}")

    miss = df_clean[present_features].isna()
    n = len(df_clean)

    per_country = pd.DataFrame({
        "country": country,
        "variable": present_features,
        "n": n,
        "n_missing": miss.sum().values,
        "pct_missing": (miss.mean() * 100).round(2).values,
    })
    missingness_records.append(per_country)

    # accumulate pooled counts (aligned by variable name, missing ones get 0)
    n_total.loc[present_features] += n
    n_missing_total.loc[present_features] += miss.sum()

# -----------------------------------------------------------------------------
# Per-country table
# -----------------------------------------------------------------------------
if missingness_records:
    missingness_by_country = pd.concat(missingness_records, ignore_index=True)
    out_path_country = os.path.join(RESULTS_DIR, "missingness_by_country.csv")
    missingness_by_country.to_csv(out_path_country, index=False)
    print(f"\nSaved per-country missingness table -> {out_path_country}")
else:
    print("\nWARNING: no per-country missingness records were generated.")

# -----------------------------------------------------------------------------
# Pooled table across all countries (the table for the manuscript / response letter)
# -----------------------------------------------------------------------------
pooled_missingness = pd.DataFrame({
    "variable": FEATURES,
    "n_total": n_total.values,
    "n_missing": n_missing_total.values,
    "pct_missing": np.where(
        n_total.values > 0,
        (n_missing_total.values / n_total.values * 100).round(2),
        np.nan
    ),
}).sort_values("pct_missing", ascending=False).reset_index(drop=True)

out_path_pooled_csv = os.path.join(RESULTS_DIR, "missingness_pooled.csv")
pooled_missingness.to_csv(out_path_pooled_csv, index=False)

# Markdown version for direct paste into manuscript / reviewer response
out_path_pooled_md = os.path.join(RESULTS_DIR, "missingness_pooled.md")
with open(out_path_pooled_md, "w") as f:
    f.write("| Variable | N | N Missing | % Missing |\n")
    f.write("|---|---|---|---|\n")
    for _, r in pooled_missingness.iterrows():
        f.write(f"| {r['variable']} | {int(r['n_total'])} | {int(r['n_missing'])} | {r['pct_missing']:.2f}% |\n")

print(f"Saved pooled missingness table -> {out_path_pooled_csv}")
print(f"Saved pooled missingness table (markdown) -> {out_path_pooled_md}")

print("\nPooled missingness (sorted, highest first):")
print(pooled_missingness.to_string(index=False))

if failed_countries:
    print(f"\nNOTE: the following countries were skipped due to errors/empty samples: {failed_countries}")

In [ ]:
# 36,704

29968 + 5567  + 5543 + 3533 + 2246 + 2239 + 1278 + 816 + 654 + 220 + 220 + 12 + 14 +  9 + 4

In [ ]:
ROOT_DIR = DATA_ROOT_DIR

baseline_df = pd.read_csv(f"{RESULTS_DIR}/anemia_model_results_by_country.csv")
baseline_df.head(20)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os

# -----------------------------------------------------------------------------
# Load results
# -----------------------------------------------------------------------------
df = pd.read_csv(f"{RESULTS_DIR}/anemia_model_results_by_country.csv")

# -----------------------------------------------------------------------------
# Reshape into long format
# -----------------------------------------------------------------------------
models = {
    "lg": "LogReg",
    "lgbm": "LightGBM",
    "xgb": "XGBoost",
    "tabpfn": "TabPFN"
}

plot_rows = []

for model_key, model_name in models.items():

    temp = pd.DataFrame({
        "country": df["country"],
        "model": model_name,
        "auc": df[f"{model_key}_auc"],
        "ci_lower": df[f"{model_key}_auc_ci_lower"],
        "ci_upper": df[f"{model_key}_auc_ci_upper"],
    })

    plot_rows.append(temp)

plot_df = pd.concat(plot_rows, ignore_index=True)

# -----------------------------------------------------------------------------
# Clean country labels
# -----------------------------------------------------------------------------
plot_df["country"] = (
    plot_df["country"]
    .str.replace("_DHS_", " ", regex=False)
    .str.replace("_", " ", regex=False)
    .str.replace(r"(\d{4}) (\d{4})", r"\1--\2", regex=True)
)

# -----------------------------------------------------------------------------
# Compute asymmetric error bars
# -----------------------------------------------------------------------------
plot_df["err_lower"] = plot_df["auc"] - plot_df["ci_lower"]
plot_df["err_upper"] = plot_df["ci_upper"] - plot_df["auc"]

# -----------------------------------------------------------------------------
# Plot
# -----------------------------------------------------------------------------
sns.set_style("whitegrid")

fig, ax = plt.subplots(figsize=(15, 6))

# Draw bars without seaborn CI
barplot = sns.barplot(
    data=plot_df,
    x="country",
    y="auc",
    hue="model",
    errorbar=None,
    ax=ax
)

# -----------------------------------------------------------------------------
# Add manual CI error bars
# -----------------------------------------------------------------------------
# -----------------------------------------------------------------------------
# Add manual CI error bars
# -----------------------------------------------------------------------------
bars = [bar for bar in ax.patches if bar.get_height() > 0]

for bar, (_, row) in zip(bars, plot_df.iterrows()):

    x = bar.get_x() + bar.get_width() / 2
    y = bar.get_height()

    ax.errorbar(
        x=x,
        y=y,
        yerr=np.array([
            [row["err_lower"]],
            [row["err_upper"]]
        ]),
        fmt="none",
        ecolor="black",
        elinewidth=1,
        capsize=3
    )

# -----------------------------------------------------------------------------
# Formatting
# -----------------------------------------------------------------------------
ax.set_title(
    "Model Performance Across Countries (AUC-ROC with 95% CI)",
    fontsize=14,
    weight="bold"
)

ax.set_ylabel("AUC-ROC")
ax.set_xlabel("Country")

plt.xticks(rotation=45, ha="right")

ax.set_ylim(0.5, 1.05)

plt.legend(title="Model", frameon=False)

plt.tight_layout()

# -----------------------------------------------------------------------------
# Save
# -----------------------------------------------------------------------------
save_path = os.path.join(
    FIGURES_DIR,
    "anemia_model_performance_auc"
)

plt.savefig(save_path + ".png", dpi=300, bbox_inches="tight")
plt.savefig(save_path + ".pdf", bbox_inches="tight")

plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os

# -----------------------------------------------------------------------------
# Load LOCO results
# -----------------------------------------------------------------------------
df = pd.read_csv(f"{RESULTS_DIR}/anemia_loco_results.csv")

# -----------------------------------------------------------------------------
# Reshape into long format
# -----------------------------------------------------------------------------
models = {
    "lg": "LogReg",
    "lgbm": "LightGBM",
    "xgb": "XGBoost",
    "tabpfn": "TabPFN"
}

plot_rows = []

for model_key, model_name in models.items():

    temp = pd.DataFrame({
        "country": df["test_country"],
        "model": model_name,
        "auc": df[f"{model_key}_auc"],
        "ci_lower": df[f"{model_key}_auc_ci_lower"],
        "ci_upper": df[f"{model_key}_auc_ci_upper"],
    })

    plot_rows.append(temp)

plot_df = pd.concat(plot_rows, ignore_index=True)

# -----------------------------------------------------------------------------
# Clean country labels
# -----------------------------------------------------------------------------
plot_df["country"] = (
    plot_df["country"]
    .str.replace("_DHS_", " ", regex=False)
    .str.replace("_", " ", regex=False)
    .str.replace(r"(\d{4}) (\d{4})", r"\1--\2", regex=True)
)

# -----------------------------------------------------------------------------
# Compute asymmetric error bars
# -----------------------------------------------------------------------------
plot_df["err_lower"] = plot_df["auc"] - plot_df["ci_lower"]
plot_df["err_upper"] = plot_df["ci_upper"] - plot_df["auc"]

# -----------------------------------------------------------------------------
# Plot
# -----------------------------------------------------------------------------
sns.set_style("whitegrid")

fig, ax = plt.subplots(figsize=(15, 6))

# Draw bars without seaborn CI
barplot = sns.barplot(
    data=plot_df,
    x="country",
    y="auc",
    hue="model",
    errorbar=None,
    ax=ax
)

# -----------------------------------------------------------------------------
# Add manual CI error bars
# -----------------------------------------------------------------------------
bars = [bar for bar in ax.patches if bar.get_height() > 0]

for bar, (_, row) in zip(bars, plot_df.iterrows()):

    x = bar.get_x() + bar.get_width() / 2
    y = bar.get_height()

    ax.errorbar(
        x=x,
        y=y,
        yerr=np.array([
            [row["err_lower"]],
            [row["err_upper"]]
        ]),
        fmt="none",
        ecolor="black",
        elinewidth=1,
        capsize=3
    )

# -----------------------------------------------------------------------------
# Formatting
# -----------------------------------------------------------------------------
ax.set_title(
    "Leave-One-Country-Out Model Performance Across Countries (AUC-ROC with 95% CI)",
    fontsize=14,
    weight="bold"
)

ax.set_ylabel("AUC-ROC")
ax.set_xlabel("Country")

plt.xticks(rotation=45, ha="right")

ax.set_ylim(0.5, 1.05)

plt.legend(title="Model", frameon=False)

plt.tight_layout()

# -----------------------------------------------------------------------------
# Save
# -----------------------------------------------------------------------------
save_path = os.path.join(
    FIGURES_DIR,
    "anemia_loco_auc"
)

plt.savefig(save_path + ".png", dpi=300, bbox_inches="tight")
plt.savefig(save_path + ".pdf", bbox_inches="tight")

plt.show()

In [ ]:
import pandas as pd

# -----------------------------------------------------------------------------
# Load LOCO results
# -----------------------------------------------------------------------------
df = pd.read_csv(f"{RESULTS_DIR}/anemia_loco_results.csv")

# -----------------------------------------------------------------------------
# Clean country names
# -----------------------------------------------------------------------------
df["test_country"] = (
    df["test_country"]
    .str.replace("_DHS_", " ", regex=False)
    .str.replace("_", " ", regex=False)
    .str.replace(r"(\d{4}) (\d{4})", r"\1--\2", regex=True)
)

# -----------------------------------------------------------------------------
# Helper function
# -----------------------------------------------------------------------------
def fmt(mean, lo, hi):
    return f"{mean:.3f} ({lo:.3f}--{hi:.3f})"

# -----------------------------------------------------------------------------
# Build LaTeX rows
# -----------------------------------------------------------------------------
latex_rows = []

for _, row in df.iterrows():

    lg = fmt(row["lg_auc"], row["lg_auc_ci_lower"], row["lg_auc_ci_upper"])
    lgbm = fmt(row["lgbm_auc"], row["lgbm_auc_ci_lower"], row["lgbm_auc_ci_upper"])
    xgb = fmt(row["xgb_auc"], row["xgb_auc_ci_lower"], row["xgb_auc_ci_upper"])
    tabpfn = fmt(row["tabpfn_auc"], row["tabpfn_auc_ci_lower"], row["tabpfn_auc_ci_upper"])

    latex_rows.append(
        f"{row['test_country']} & "
        f"{lg} & "
        f"{lgbm} & "
        f"{xgb} & "
        f"{tabpfn} \\\\"
    )

# -----------------------------------------------------------------------------
# Print LaTeX table
# -----------------------------------------------------------------------------
print(r"\begin{table*}[htbp]")
print(r"\centering")
print(r"\caption{Leave-One-Country-Out (LOCO) model performance reported as AUC-ROC with 95\% bootstrap BCa confidence intervals.}")
print(r"\label{tab:loco_auc_results}")
print(r"\resizebox{\textwidth}{!}{")
print(r"\begin{tabular}{lcccc}")
print(r"\toprule")
print(r"\textbf{Held-out Country} & \textbf{LogReg} & \textbf{LightGBM} & \textbf{XGBoost} & \textbf{TabPFN} \\")
print(r"\midrule")

for line in latex_rows:
    print(line)

print(r"\bottomrule")
print(r"\end{tabular}")
print(r"}")
print(r"\end{table*}")

In [ ]:
import pandas as pd

# -----------------------------------------------------------------------------
# Load results
# -----------------------------------------------------------------------------
df = pd.read_csv(f"{RESULTS_DIR}/anemia_model_results_by_country.csv")

# -----------------------------------------------------------------------------
# Clean country names
# -----------------------------------------------------------------------------
df["country"] = (
    df["country"]
    .str.replace("_DHS_", " ", regex=False)
    .str.replace("_", " ", regex=False)
    .str.replace(r"(\d{4}) (\d{4})", r"\1--\2", regex=True)
)

# -----------------------------------------------------------------------------
# Helper function
# -----------------------------------------------------------------------------
def fmt(mean, lo, hi):
    return f"{mean:.3f} ({lo:.3f}--{hi:.3f})"

# -----------------------------------------------------------------------------
# Build LaTeX rows
# -----------------------------------------------------------------------------
latex_rows = []

for _, row in df.iterrows():

    lg = fmt(
        row["lg_auc"],
        row["lg_auc_ci_lower"],
        row["lg_auc_ci_upper"]
    )

    lgbm = fmt(
        row["lgbm_auc"],
        row["lgbm_auc_ci_lower"],
        row["lgbm_auc_ci_upper"]
    )

    xgb = fmt(
        row["xgb_auc"],
        row["xgb_auc_ci_lower"],
        row["xgb_auc_ci_upper"]
    )

    tabpfn = fmt(
        row["tabpfn_auc"],
        row["tabpfn_auc_ci_lower"],
        row["tabpfn_auc_ci_upper"]
    )

    latex_rows.append(
        f"{row['country']} & "
        f"{lg} & "
        f"{lgbm} & "
        f"{xgb} & "
        f"{tabpfn} \\\\"
    )

# -----------------------------------------------------------------------------
# Print final LaTeX table
# -----------------------------------------------------------------------------
print(r"\begin{table*}[htbp]")
print(r"\centering")
print(r"\caption{Model performance across countries reported as AUC-ROC with 95\% bootstrap BCa confidence intervals.}")
print(r"\label{tab:auc_results}")
print(r"\resizebox{\textwidth}{!}{")
print(r"\begin{tabular}{lcccc}")
print(r"\toprule")
print(r"\textbf{Country} & \textbf{LogReg} & \textbf{LightGBM} & \textbf{XGBoost} & \textbf{TabPFN} \\")
print(r"\midrule")

for line in latex_rows:
    print(line)

print(r"\bottomrule")
print(r"\end{tabular}")
print(r"}")
print(r"\end{table*}")

In [ ]:
shap_df = pd.read_csv(f"{RESULTS_DIR}/anemia_model_results_by_country_shap.csv")
# summary = shap_df.groupby(["model", "feature"])["importance"].mean().reset_index()
agg = shap_df.groupby(["model", "feature"]).agg(
    mean_importance=("importance", "mean"),
    std_importance=("importance", "std")
).reset_index()


feature_order = (
    agg.groupby("feature")["mean_importance"]
    .mean()
    .sort_values(ascending=True)
    .index
)

import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11
})

models = ["LogReg", "LightGBM", "XGBoost", "TabPFN"]

fig, axes = plt.subplots(
    2, 2,
    figsize=(14, 10),
    dpi=300,
    constrained_layout=True
)

axes = axes.flatten()

for i, model in enumerate(models):

    sub = agg[agg["model"] == model].copy()
    sub["feature"] = pd.Categorical(sub["feature"], categories=feature_order, ordered=True)
    sub = sub.sort_values("feature")

    axes[i].barh(
        sub["feature"],
        sub["mean_importance"],
        xerr=sub["std_importance"],
        color="#2C3E50",
        alpha=0.85,
        capsize=3
    )

    axes[i].set_title(model, loc="left", fontweight="bold")
    axes[i].set_xlabel("Mean |SHAP importance|")

    # clean style (Lancet standard)
    axes[i].spines["top"].set_visible(False)
    axes[i].spines["right"].set_visible(False)

    if i % 2 == 1:
        axes[i].set_ylabel("")
    else:
        axes[i].set_ylabel("Feature")

plt.savefig(
    f"{FIGURES_DIR}/feature_importance_plot.png",
    dpi=600,
    bbox_inches="tight"
)
plt.savefig(
    f"{FIGURES_DIR}/feature_importance_plot.pdf",
    dpi=600,
    bbox_inches="tight"
)
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

shap_df = pd.read_csv(f"{RESULTS_DIR}/anemia_model_results_by_country_shap.csv")

def clean_country(name):
    return (
        str(name)
        .replace("_DHS_", " ")
        .replace("_", " ")
        .replace("  ", " ")
        .strip()
        .replace(r"(\d{4}) (\d{4})", r"\1--\2")
    )

# If your country column has a different name, change it here
country_col = "country"

# Aggregate within each country, model, and feature
agg = shap_df.groupby([country_col, "model", "feature"]).agg(
    mean_importance=("importance", "mean"),
    std_importance=("importance", "std")
).reset_index()

models = ["LogReg", "LightGBM", "XGBoost", "TabPFN"]

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11
})

# Global feature order across all countries, for consistency
feature_order = (
    agg.groupby("feature")["mean_importance"]
    .mean()
    .sort_values(ascending=True)
    .index
)

countries = agg[country_col].dropna().unique()

for country in countries:
    fig, axes = plt.subplots(
        2, 2,
        figsize=(14, 10),
        dpi=300,
        constrained_layout=True
    )
    axes = axes.flatten()

    country_data = agg[agg[country_col] == country].copy()

    for i, model in enumerate(models):
        sub = country_data[country_data["model"] == model].copy()
        sub["feature"] = pd.Categorical(sub["feature"], categories=feature_order, ordered=True)
        sub = sub.sort_values("feature")

        axes[i].barh(
            sub["feature"],
            sub["mean_importance"],
            xerr=sub["std_importance"],
            color="#2C3E50",
            alpha=0.85,
            capsize=3
        )

        axes[i].set_title(model, loc="left", fontweight="bold")
        axes[i].set_xlabel("Mean |SHAP importance|")

        axes[i].spines["top"].set_visible(False)
        axes[i].spines["right"].set_visible(False)

        if i % 2 == 1:
            axes[i].set_ylabel("")
        else:
            axes[i].set_ylabel("Feature")



    fig.suptitle(clean_country(country), x=0.01, y=1.02, ha="left", fontsize=14, fontweight="bold")

    safe_country = str(country).replace(" ", "_").replace("/", "_")
    plt.savefig(
        f"{FIGURES_DIR}/feature_importance_plot_{safe_country}.png",
        dpi=600,
        bbox_inches="tight"
    )
    plt.savefig(
        f"{FIGURES_DIR}/feature_importance_plot_{safe_country}.pdf",
        dpi=600,
        bbox_inches="tight"
    )
    plt.show()
    plt.close(fig)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

shap_df = pd.read_csv(f"{RESULTS_DIR}/anemia_model_results_by_country_shap.csv")

country_col = "country"

# Filter only TabPFN
tabpfn_df = shap_df[shap_df["model"] == "TabPFN"].copy()

# Aggregate per country + feature
agg = tabpfn_df.groupby([country_col, "feature"]).agg(
    mean_importance=("importance", "mean"),
    std_importance=("importance", "std")
).reset_index()

# Global feature order (consistent across all countries)
feature_order = (
    agg.groupby("feature")["mean_importance"]
    .mean()
    .sort_values(ascending=True)
    .index
)

countries = sorted(agg[country_col].dropna().unique())

# Ensure exactly 16 (or adjust if needed)
countries = countries[:16]

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9
})

fig, axes = plt.subplots(
    4, 4,
    figsize=(16, 14),
    dpi=300,
    constrained_layout=True
)

axes = axes.flatten()

for i, country in enumerate(countries):
    ax = axes[i]
    sub = agg[agg[country_col] == country].copy()

    sub["feature"] = pd.Categorical(
        sub["feature"],
        categories=feature_order,
        ordered=True
    )
    sub = sub.sort_values("feature")

    ax.barh(
        sub["feature"],
        sub["mean_importance"],
        xerr=sub["std_importance"],
        color="#2C3E50",
        alpha=0.85,
        capsize=2
    )

    ax.set_title(clean_country(country), fontsize=12, loc="left", fontweight="bold")
    ax.set_xlabel("SHAP importance",fontsize=12)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # reduce clutter
    ax.tick_params(axis='y', labelsize=7)
    # Increase axis tick label sizes (this is what you want)
    # ax.tick_params(axis='x', labelsize=11)
    # ax.tick_params(axis='y', labelsize=10)

# Turn off any empty subplots (if <16 countries)
for j in range(len(countries), 16):
    axes[j].axis("off")

plt.savefig(
    f"{FIGURES_DIR}/tabpfn_feature_importance_countries.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    f"{FIGURES_DIR}/tabpfn_feature_importance_countries.pdf",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
shap_df["model"].unique()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

shap_df = pd.read_csv(f"{RESULTS_DIR}/anemia_model_results_by_country_shap.csv")

country_col = "country"

modelstr = 'XGBoost'
# Filter only the model
tabpfn_df = shap_df[shap_df["model"] == modelstr].copy()

# Aggregate per country + feature
agg = tabpfn_df.groupby([country_col, "feature"]).agg(
    mean_importance=("importance", "mean"),
    std_importance=("importance", "std")
).reset_index()

# Global feature order (consistent across all countries)
feature_order = (
    agg.groupby("feature")["mean_importance"]
    .mean()
    .sort_values(ascending=True)
    .index
)

countries = sorted(agg[country_col].dropna().unique())

# Ensure exactly 16 (or adjust if needed)
countries = countries[:16]

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9
})

fig, axes = plt.subplots(
    4, 4,
    figsize=(16, 14),
    dpi=300,
    constrained_layout=True
)

axes = axes.flatten()

for i, country in enumerate(countries):
    ax = axes[i]
    sub = agg[agg[country_col] == country].copy()

    sub["feature"] = pd.Categorical(
        sub["feature"],
        categories=feature_order,
        ordered=True
    )
    sub = sub.sort_values("feature")

    ax.barh(
        sub["feature"],
        sub["mean_importance"],
        xerr=sub["std_importance"],
        color="#2C3E50",
        alpha=0.85,
        capsize=2
    )

    ax.set_title(clean_country(country), fontsize=12, loc="left", fontweight="bold")
    ax.set_xlabel("SHAP importance",fontsize=12)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # reduce clutter
    ax.tick_params(axis='y', labelsize=7)
    # Increase axis tick label sizes (this is what you want)
    # ax.tick_params(axis='x', labelsize=11)
    # ax.tick_params(axis='y', labelsize=10)

# Turn off any empty subplots (if <16 countries)
for j in range(len(countries), 16):
    axes[j].axis("off")

plt.savefig(
    f"{FIGURES_DIR}/{modelstr}_feature_importance_countries.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    f"{FIGURES_DIR}/{modelstr}_feature_importance_countries.pdf",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:

df = pd.read_csv(f"{RESULTS_DIR}/dhs_country_data_summary.csv")

# ---- CLEAN COUNTRY COLUMN ----
df["country"] = (
    df["country"]
    .str.replace("_DHS_", " ", regex=False)
    .str.replace("_", " ", regex=False)
    .str.replace(r"(\d{4}) (\d{4})", r"\1--\2", regex=True)
)

df_latex = df[[
    "country",
    "raw_total",
    "processed_total",
    "mean_age_months",
    "raw_male_total",
    "raw_female_total",
    "processed_male",
    "processed_female"
]].copy()

df_latex.columns = [
    "Country",
    "N Raw",
    "N Final",
    "Mean Age (months)",
    "Raw Male",
    "Raw Female",
    "Final Male",
    "Final Female"
]

# Weighted mean age using processed sample size
weighted_mean_age = (
    (df["mean_age_months"] * df["processed_total"]).sum()
    / df["processed_total"].sum()
)

total_row = {
    "Country": r"\textbf{Total}",
    "N Raw": df["raw_total"].sum(),
    "N Final": df["processed_total"].sum(),
    "Mean Age (months)": round(weighted_mean_age, 2),
    "Raw Male": df["raw_male_total"].sum(),
    "Raw Female": df["raw_female_total"].sum(),
    "Final Male": df["processed_male"].sum(),
    "Final Female": df["processed_female"].sum()
}

df_latex = pd.concat([df_latex, pd.DataFrame([total_row])], ignore_index=True)

latex_table = df_latex.to_latex(
    index=False,
    escape=False,
    column_format="lrrrrrrr",
    float_format="%.2f"
)

print(latex_table)


In [ ]:
df = pd.read_csv(f"{RESULTS_DIR}/exclusion_counts_by_country.csv")

df["country"] = (
    df["country"]
    .str.replace("_DHS_", " ", regex=False)
    .str.replace("_", " ", regex=False)
    .str.replace(r"(\d{4}) (\d{4})", r"\1--\2", regex=True)
)

df_fmt = df.copy()

# optional: add thousands separators for readability
for col in df_fmt.columns[1:]:
    df_fmt[col] = df_fmt[col].map(lambda x: f"{int(x):,}")

latex_table = df_fmt.to_latex(
    index=False,
    escape=False,
    column_format="lrrrrrr"
)

print(latex_table) 

In [ ]:
rev_df = pd.read_csv(f"{RESULTS_DIR}/anemia_reverse_loco_results.csv")



rev_df["test_country"] = rev_df["test_country"].str.replace("_", " ", regex=False)

rev_df["train_country"] = rev_df["train_country"].str.replace("_", " ", regex=False)

pivot_lg = rev_df.pivot_table(
    index="train_country", columns="test_country", values="lg_auc"
)

pivot_lgbm = rev_df.pivot_table(
    index="train_country", columns="test_country", values="lgbm_auc"
)

pivot_xgb = rev_df.pivot_table(
    index="train_country", columns="test_country", values="xgb_auc"
)

pivot_tabpfn = rev_df.pivot_table(
    index="train_country", columns="test_country", values="tabpfn_auc"
)


model_pivots = {
    "LogReg": pivot_lg,
    "LightGBM": pivot_lgbm,
    "XGBoost": pivot_xgb,
    "TabPFN": pivot_tabpfn
}


import re
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# --------------------------------------
# STYLE (Lancet-like)
# --------------------------------------
sns.set_style("white")

plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10
})

# --------------------------------------
# Figure (more breathing room)
# --------------------------------------
fig = plt.figure(figsize=(18, 15), dpi=300)

gs = fig.add_gridspec(
    2, 3,
    width_ratios=[1, 1, 0.04],
    wspace=0.45,   # ✅ more horizontal space
    hspace=0.5     # ✅ more vertical space
)

axes = [
    fig.add_subplot(gs[0, 0]),
    fig.add_subplot(gs[0, 1]),
    fig.add_subplot(gs[1, 0]),
    fig.add_subplot(gs[1, 1]),
]

cbar_ax = fig.add_subplot(gs[:, 2])

panel_labels = ["A", "B", "C", "D"]

# --------------------------------------
# Color scale
# --------------------------------------
vmin = rev_df[["lg_auc","lgbm_auc","xgb_auc","tabpfn_auc"]].min().min()
vmax = rev_df[["lg_auc","lgbm_auc","xgb_auc","tabpfn_auc"]].max().max()

# --------------------------------------
# Country label cleaner
# --------------------------------------
def clean_country(name):
    name = name.replace("_", " ").replace("DHS", "")
    name = re.sub(r"\s+", " ", name)
    name = re.sub(r"(\d{4})\s+(\d{4})", r"\1–\2", name)  # en dash
    return name.strip()

# --------------------------------------
# Plot
# --------------------------------------
for i, (ax, (name, pivot)) in enumerate(zip(axes, model_pivots.items())):

    # Ensure square + aligned
    pivot = pivot.copy()
    pivot = pivot.loc[pivot.index.intersection(pivot.columns),
                      pivot.index.intersection(pivot.columns)]
    pivot = pivot.sort_index()
    pivot = pivot[pivot.index]

    # Clean labels
    pivot.index = [clean_country(x) for x in pivot.index]
    pivot.columns = [clean_country(x) for x in pivot.columns]

    mask = np.eye(pivot.shape[0], dtype=bool)

    sns.heatmap(
        pivot,
        ax=ax,
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        mask=mask,
        annot=True,
        fmt=".2f",
        annot_kws={"size": 5},
        linewidths=0.2,
        linecolor="white",
        cbar=(i == 0),
        cbar_ax=cbar_ax if i == 0 else None
    )

    # Titles
    ax.set_title(name, loc="left", fontweight="bold", pad=8)
    ax.set_xlabel("Test country")

    if i % 2 == 1:
        ax.set_ylabel("")
        ax.set_yticklabels([])
    else:
        ax.set_ylabel("Train country")

    # Tick formatting
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

    # Remove spines
    for spine in ax.spines.values():
        spine.set_visible(False)

    # Subtle grid (Lancet-style)
    ax.set_axisbelow(True)
    ax.grid(which="major", linestyle=":", linewidth=0.3, alpha=0.3)

    # Panel label
    ax.text(
        -0.14, 1.06,
        panel_labels[i],
        transform=ax.transAxes,
        fontsize=13,
        fontweight="bold"
    )

# --------------------------------------
# Colorbar
# --------------------------------------
cbar_ax.set_ylabel("AUC–ROC", fontsize=11)
cbar_ax.tick_params(labelsize=8)

# --------------------------------------
# Save
# --------------------------------------
plt.savefig(
    f"{FIGURES_DIR}/reverse_loco_plot.pdf",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
"""
reverse_loco_network_plot.py
────────────────────────────────────────────────────────────────────
Replaces the heatmap version of the Reverse-LOCO figure.
Drop-in: same inputs (rev_df + FIGURES_DIR) → PDF + PNG output.

Network layout:
  • Nodes   = countries arranged on a unit circle
  • Edges   = directed arc from TRAIN country  →  TEST country
              ** only drawn when AUC ≥ AUC_THRESHOLD **
  • Colour  = AUC-ROC (viridis, shared scale across all four panels)
  • Width   = scaled to AUC-ROC (thin ≈ low, thick ≈ high)
  • Size    = node size ∝ out-degree (# edges above threshold)
  • Panels  = A LogReg | B LightGBM | C XGBoost | D TabPFN

Adjust AUC_THRESHOLD below to control edge density.
"""

import re
import math
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
import networkx as nx

# ─────────────────────────────────────────────────
# 0.  Paths  (edit to match your project)
# ─────────────────────────────────────────────────
RESULTS_DIR  = "results"
FIGURES_DIR  = "figures"

# ─────────────────────────────────────────────────
# 0b. Threshold  ← adjust to control edge density
# ─────────────────────────────────────────────────
AUC_THRESHOLD = 0.65   # only edges with AUC ≥ this are drawn

# ─────────────────────────────────────────────────
# 1.  Load & tidy data
# ─────────────────────────────────────────────────
rev_df = pd.read_csv(f"{RESULTS_DIR}/anemia_reverse_loco_results.csv")
rev_df["test_country"]  = rev_df["test_country"].str.replace("_", " ", regex=False)
rev_df["train_country"] = rev_df["train_country"].str.replace("_", " ", regex=False)


def clean_country(name: str) -> str:
    """Normalise country labels: drop 'DHS', collapse spaces, en-dash year ranges."""
    name = name.replace("_", " ").replace("DHS", "")
    name = re.sub(r"\s+", " ", name)
    name = re.sub(r"(\d{4})\s+(\d{4})", r"\1–\2", name)
    return name.strip()


def make_pivot(col: str) -> pd.DataFrame:
    piv = rev_df.pivot_table(index="train_country", columns="test_country", values=col)
    piv = piv.loc[piv.index.intersection(piv.columns),
                  piv.index.intersection(piv.columns)].sort_index()
    piv = piv[piv.index]
    piv.index   = [clean_country(x) for x in piv.index]
    piv.columns = [clean_country(x) for x in piv.columns]
    return piv


model_pivots = {
    "Logistic Regression": make_pivot("lg_auc"),
    "LightGBM":            make_pivot("lgbm_auc"),
    "XGBoost":             make_pivot("xgb_auc"),
    "TabPFN":              make_pivot("tabpfn_auc"),
}

auc_cols = ["lg_auc", "lgbm_auc", "xgb_auc", "tabpfn_auc"]
VMIN = rev_df[auc_cols].min().min()
VMAX = rev_df[auc_cols].max().max()

# ─────────────────────────────────────────────────
# 2.  Global style
# ─────────────────────────────────────────────────
matplotlib.rcParams.update({
    "font.family":      "DejaVu Sans",
    "font.size":        9,
    "axes.titlesize":   11,
    "axes.labelsize":   9,
    "pdf.fonttype":     42,   # embeddable fonts for Nature submission
    "ps.fonttype":      42,
})

CMAP  = cm.viridis
NORM  = mcolors.Normalize(vmin=VMIN, vmax=VMAX)

# Edge width mapping: AUC → linewidth  (0.4 … 2.8)
def auc_to_lw(auc: float) -> float:
    t = (auc - VMIN) / max(VMAX - VMIN, 1e-9)
    return 0.4 + t * 2.4

# ─────────────────────────────────────────────────
# 3.  Figure layout  (2 × 2 panels + right colorbar)
# ─────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 14), dpi=300)
gs  = fig.add_gridspec(
    2, 3,
    width_ratios=[1, 1, 0.03],
    left=0.04, right=0.92,
    top=0.93,  bottom=0.05,
    wspace=0.14, hspace=0.22,
)
axes = [
    fig.add_subplot(gs[0, 0]),
    fig.add_subplot(gs[0, 1]),
    fig.add_subplot(gs[1, 0]),
    fig.add_subplot(gs[1, 1]),
]
cbar_ax = fig.add_subplot(gs[:, 2])

PANEL_LABELS = ["A", "B", "C", "D"]

# ─────────────────────────────────────────────────
# 4.  Draw one network panel
# ─────────────────────────────────────────────────

def circular_positions(countries: list[str], radius: float = 1.0) -> dict:
    """Unit-circle positions, starting from the top (π/2), clockwise."""
    n = len(countries)
    pos = {}
    for k, c in enumerate(countries):
        angle = math.pi / 2 - 2 * math.pi * k / n   # clockwise from top
        pos[c] = (radius * math.cos(angle), radius * math.sin(angle))
    return pos


def draw_network_panel(ax: plt.Axes, pivot: pd.DataFrame, title: str, panel_lbl: str):
    countries = list(pivot.index)
    n         = len(countries)
    pos       = circular_positions(countries)

    ax.set_xlim(-1.72, 1.72)
    ax.set_ylim(-1.72, 1.72)
    ax.set_aspect("equal")
    ax.axis("off")

    # ── 4a.  Background disc (subtle) ──────────────────
    bg = plt.Circle((0, 0), 1.12, color="#f7f7f7", zorder=0)
    ax.add_patch(bg)

    # ── 4b.  Draw edges (only AUC ≥ AUC_THRESHOLD) ────
    NODE_R     = 0.09
    edge_count = 0

    for train_c in countries:
        for test_c in countries:
            if train_c == test_c:
                continue
            auc = pivot.loc[train_c, test_c]
            if pd.isna(auc) or auc < AUC_THRESHOLD:
                continue
            edge_count += 1

            colour = CMAP(NORM(auc))
            lw     = auc_to_lw(auc)
            alpha  = 0.55 + 0.40 * (auc - AUC_THRESHOLD) / max(VMAX - AUC_THRESHOLD, 1e-9)

            x0, y0 = pos[train_c]
            x1, y1 = pos[test_c]
            dx, dy = x1 - x0, y1 - y0
            dist   = math.hypot(dx, dy)
            if dist < 1e-6:
                continue
            sh     = NODE_R / dist
            xs, ys = x0 + sh * dx, y0 + sh * dy
            xe, ye = x1 - sh * dx, y1 - sh * dy

            ax.annotate(
                "",
                xy=(xe, ye), xytext=(xs, ys),
                arrowprops=dict(
                    arrowstyle="-|>",
                    color=colour,
                    lw=lw,
                    alpha=alpha,
                    connectionstyle="arc3,rad=0.22",
                    mutation_scale=9,
                ),
                zorder=2,
            )

    # ── 4c.  Draw nodes (size ∝ out-degree) ───────────
    degree  = {c: 0 for c in countries}
    for train_c in countries:
        for test_c in countries:
            if train_c == test_c:
                continue
            auc = pivot.loc[train_c, test_c]
            if not pd.isna(auc) and auc >= AUC_THRESHOLD:
                degree[train_c] += 1
    max_deg = max(degree.values()) if max(degree.values()) > 0 else 1

    for c, (x, y) in pos.items():
        sz = 220 + 180 * degree[c] / max_deg
        ax.scatter(x, y, s=sz + 80, color="#bbbbbb", zorder=3, linewidths=0)
        ax.scatter(x, y, s=sz,      color="white",   zorder=4,
                   edgecolors="#2c2c2c", linewidths=1.5)

    # ── 4d.  Country labels ───────────────────────────
    LABEL_R = 1.28
    for k, c in enumerate(countries):
        angle  = math.pi / 2 - 2 * math.pi * k / n
        lx     = LABEL_R * math.cos(angle)
        ly     = LABEL_R * math.sin(angle)
        ha     = "center"
        if math.cos(angle) >  0.12:  ha = "left"
        if math.cos(angle) < -0.12:  ha = "right"
        va     = "center"
        if math.sin(angle) >  0.6:   va = "bottom"
        if math.sin(angle) < -0.6:   va = "top"

        # Wrap long labels
        words = c.split()
        if len(words) > 2:
            mid   = math.ceil(len(words) / 2)
            label = " ".join(words[:mid]) + "\n" + " ".join(words[mid:])
        else:
            label = c

        ax.text(
            lx, ly, label,
            ha=ha, va=va,
            fontsize=6.8,
            fontweight="semibold",
            color="#1a1a1a",
            linespacing=1.2,
            zorder=6,
        )

        # Short spoke line from node surface to label area
        node_x, node_y = pos[c]
        spoke_r0 = 0.12
        spoke_r1 = 1.18
        ax.plot(
            [node_x + spoke_r0 * math.cos(angle), spoke_r1 * math.cos(angle)],
            [node_y + spoke_r0 * math.sin(angle), spoke_r1 * math.sin(angle)],
            color="#cccccc", lw=0.55, zorder=1,
        )

    # ── 4e.  Panel title & label ──────────────────────
    ax.text(
        0.50, 1.08, title,
        transform=ax.transAxes,
        ha="center", va="bottom",
        fontsize=11, fontweight="bold", color="#1a1a1a",
    )
    ax.text(
        -0.03, 1.10, panel_lbl,
        transform=ax.transAxes,
        fontsize=14, fontweight="bold", color="#1a1a1a",
    )

    # ── 4f.  Node-count annotation ─────────────────────
    ax.text(
        0.50, -0.06,
        f"n = {n} countries",
        transform=ax.transAxes,
        ha="center", fontsize=7.5, color="#555555", style="italic",
    )


# ─────────────────────────────────────────────────
# 5.  Render all four panels
# ─────────────────────────────────────────────────
for ax, (name, pivot), lbl in zip(axes, model_pivots.items(), PANEL_LABELS):
    draw_network_panel(ax, pivot, name, lbl)

# ─────────────────────────────────────────────────
# 6.  Shared vertical colorbar
# ─────────────────────────────────────────────────
sm = cm.ScalarMappable(cmap=CMAP, norm=NORM)
sm.set_array([])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation="vertical")
cbar.set_label("AUC–ROC", fontsize=10, labelpad=8)
cbar.ax.tick_params(labelsize=8)
cbar.outline.set_linewidth(0.5)

# Tick marks at meaningful AUC thresholds
ticks = np.round(np.linspace(VMIN, VMAX, 6), 2)
cbar.set_ticks(ticks)
cbar.set_ticklabels([f"{t:.2f}" for t in ticks])

# ─────────────────────────────────────────────────
# 7.  Figure-level legend (edge semantics)
# ─────────────────────────────────────────────────
legend_elements = [
    Line2D([0], [0], color=CMAP(NORM(VMAX)), lw=2.2,
           label=f"Higher AUC  (≥ {VMAX:.2f})"),
    Line2D([0], [0], color=CMAP(NORM((VMIN + VMAX) / 2)), lw=1.3,
           label=f"Moderate AUC (≈ {(VMIN+VMAX)/2:.2f})"),
    Line2D([0], [0], color=CMAP(NORM(VMIN)), lw=0.5,
           label=f"Lower AUC   (≤ {VMIN:.2f})"),
    Line2D([0], [0], color="none",
           marker=">", markerfacecolor="#555", markersize=6,
           label="Direction: train → test country"),
]
fig.legend(
    handles=legend_elements,
    loc="lower center",
    ncol=4,
    fontsize=8,
    frameon=True,
    framealpha=0.9,
    edgecolor="#cccccc",
    bbox_to_anchor=(0.47, -0.02),
    handlelength=2.2,
    handleheight=1.0,
)

# ─────────────────────────────────────────────────
# 8.  Super-title
# ─────────────────────────────────────────────────
# fig.suptitle(
#     "Cross-country generalisation of anaemia prediction models — Reverse LOCO evaluation",
#     fontsize=12,
#     fontweight="bold",
#     color="#1a1a1a",
#     y=0.97,
# )

# fig.text(
#     0.45, 0.93,
#     "Directed edges represent model transfer from training country to test country; "
#     "edge colour and width encode AUC–ROC performance.",
#     ha="center", fontsize=8, color="#555555", style="italic",
# )

# ─────────────────────────────────────────────────
# 9.  Save
# ─────────────────────────────────────────────────
# for ext in ("pdf", "png"):
fig.savefig(
    f"{FIGURES_DIR}/reverse_loco_network_plot.pdf",
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)
print(f"Saved → {FIGURES_DIR}/reverse_loco_network_plot.pdf")

plt.show()

In [ ]:
# !pip install netgraph


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import numpy as np
import re

# --------------------------------------
# Load data
# --------------------------------------
files = glob.glob(f"{RESULTS_DIR}/*_fewshot_results.csv")
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

countries = sorted(df["country"].unique())

# --------------------------------------
# Style
# --------------------------------------
sns.set_style("white")

palette = {
    "LogReg":   "#4D4D4D",
    "LightGBM": "#1f77b4",
    "XGBoost":  "#ff7f0e",
    "TabPFN":   "#8B0000"
}

# --------------------------------------
# Figure layout (IMPORTANT: no shared axes)
# --------------------------------------
n_cols = 4
n_rows = int(np.ceil(len(countries) / n_cols))

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(18, 9),
    sharex=False,
    sharey=True
)

axes = axes.flatten()

# --------------------------------------
# Plot per country
# --------------------------------------
for i, country in enumerate(countries):
    ax = axes[i]
    cdf = df[df["country"] == country]

    for model in ["LogReg", "LightGBM", "XGBoost", "TabPFN"]:
        mdf = cdf[cdf["model"] == model].sort_values("n_shot")

        if mdf.empty:
            continue

        color = palette[model]

        ax.plot(
            mdf["n_shot"],
            mdf["auc"],
            marker="o",
            linewidth=2.5 if model == "TabPFN" else 1.6,
            color=color,
            label=model
        )

        # ── CIs: shaded band around the AUC line ──────────────────────────
        ax.fill_between(
            mdf["n_shot"],
            mdf["auc_ci_lower"],
            mdf["auc_ci_upper"],
            color=color,
            alpha=0.12,       # subtle — keeps the plot readable
            linewidth=0,      # no border on the band
        )

    label = country.replace("_", " ").replace("DHS", "")
    label = re.sub(r"(\d{4})\s+(\d{4})", r"\1 -- \2", label)
    ax.set_title(label, fontsize=10)

    # Clean spines
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)

    ax.tick_params(axis='both', labelsize=8)

    # Only show x-label on bottom row
    if i >= (n_rows - 1) * n_cols:
        ax.set_xlabel("n-shot (training samples)", fontsize=9)

# --------------------------------------
# Remove empty subplots
# --------------------------------------
for j in range(len(countries), len(axes)):
    fig.delaxes(axes[j])

# --------------------------------------
# Shared Y label (FIXED POSITION)
# --------------------------------------
fig.text(
    0.01, 0.5,
    "AUC-ROC",
    va="center",
    rotation="vertical",
    fontsize=12
)

# --------------------------------------
# Legend (global, clean placement)
# --------------------------------------
handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=4,
    frameon=False,
    fontsize=10
)

# --------------------------------------
# Layout tuning (VERY IMPORTANT)
# --------------------------------------
plt.subplots_adjust(
    left=0.06,
    right=0.98,
    top=0.88,
    bottom=0.10,
    hspace=0.35,
    wspace=0.25
)

# --------------------------------------
# Save high-quality outputs
# --------------------------------------
plt.savefig(f"{FIGURES_DIR}/fewshot_auc_by_country_clean.pdf", bbox_inches="tight")
plt.savefig(f"{FIGURES_DIR}/fewshot_auc_by_country_clean.tiff", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re

df = pd.read_csv(f"{RESULTS_DIR}/subgroup_analysis_optuna_models.csv")

# --------------------------------------
# STYLE (Lancet-like)
# --------------------------------------
sns.set_style("white")

plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 12,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11
})


palette = {
    "LogReg": "#4D4D4D",
    "LightGBM": "#1f77b4",
    "XGBoost": "#ff7f0e",
    "TabPFN": "#8B0000"
}

countries = sorted(df["country"].unique())

# --------------------------------------
# CLEAN LABELS
# --------------------------------------
def clean_country(name):
    name = name.replace("_", " ").replace("DHS", "")
    name = re.sub(r"\s+", " ", name)
    name = re.sub(r"(\d{4})\s+(\d{4})", r"\1–\2", name)
    return name.strip()

def clean_subgroup(name):
    return name.replace("_", " ").capitalize()

df["subgroup"] = df["subgroup"].apply(clean_subgroup)

# --------------------------------------
# 4x4 GRID LAYOUT
# --------------------------------------
n_cols = 4
n_rows = 4  # fixed since we know 16 countries

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(22, 18),   # balanced square layout
    sharey=True
)

axes = axes.flatten()

# --------------------------------------
# PLOT
# --------------------------------------
for i, country in enumerate(countries):

    ax = axes[i]
    cdf = df[df["country"] == country]

    plot_df = (
        cdf.groupby(["subgroup", "model"], as_index=False)["auc"]
        .mean()
    )

    sns.barplot(
        data=plot_df,
        x="subgroup",
        y="auc",
        hue="model",
        ax=ax,
        palette=palette,
        edgecolor="black",
        linewidth=0.3
    )

    # Title
    ax.set_title(clean_country(country), fontweight="bold", pad=6)

    ax.set_ylim(0.5, 1.0)

    ax.set_xlabel("")

    # Reduce clutter
    if i % n_cols != 0:
        ax.set_ylabel("")
        ax.set_yticklabels([])
    else:
        ax.set_ylabel("AUC–ROC")

    ax.tick_params(axis="x", rotation=30)

    # Lancet styling
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.set_axisbelow(True)
    ax.grid(axis="y", linestyle=":", linewidth=0.4, alpha=0.5)

    # remove per-plot legend
    if ax.get_legend():
        ax.get_legend().remove()

# --------------------------------------
# REMOVE EMPTY AXES (safety)
# --------------------------------------
for j in range(len(countries), len(axes)):
    fig.delaxes(axes[j])

# --------------------------------------
# GLOBAL LEGEND
# --------------------------------------
handles, labels = axes[0].get_legend_handles_labels()

legend = fig.legend(
    handles,
    labels,
    title="Model",
    loc="lower center",
    ncol=4,
    frameon=False,
    fontsize=11,
    title_fontsize=13,
    bbox_to_anchor=(0.5, -0.02)
)

legend.get_title().set_fontweight("bold")
# --------------------------------------
# GLOBAL LABELS
# --------------------------------------
fig.supylabel("AUC–ROC", fontsize=13)
fig.supxlabel("Subgroup", fontsize=13)

# --------------------------------------
# SPACING (critical for 4x4 grid)
# --------------------------------------
plt.subplots_adjust(
    left=0.06,
    right=0.98,
    top=0.94,
    bottom=0.10,
    hspace=0.55,
    wspace=0.30
)

# --------------------------------------
# SAVE (journal-ready)
# --------------------------------------
plt.savefig(
    f"{FIGURES_DIR}/subgroup_grouped_bar_by_country.pdf",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re

df = pd.read_csv(f"{RESULTS_DIR}/subgroup_analysis_optuna_models.csv")

# --------------------------------------
# STYLE (Lancet-like)
# --------------------------------------
sns.set_style("white")

plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 12,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11
})

palette = {
    "LogReg":   "#4D4D4D",
    "LightGBM": "#1f77b4",
    "XGBoost":  "#ff7f0e",
    "TabPFN":   "#8B0000"
}

countries  = sorted(df["country"].unique())
subgroups  = sorted(df["subgroup"].unique())   # after clean applied below

# --------------------------------------
# CLEAN LABELS
# --------------------------------------
def clean_country(name):
    name = name.replace("_", " ").replace("DHS", "")
    name = re.sub(r"\s+", " ", name)
    name = re.sub(r"(\d{4})\s+(\d{4})", r"\1–\2", name)
    return name.strip()

def clean_subgroup(name):
    return name.replace("_", " ").capitalize()

df["subgroup"] = df["subgroup"].apply(clean_subgroup)
subgroups = sorted(df["subgroup"].unique())

# --------------------------------------
# VALUE LABEL MAPS  (optional cosmetic)
# --------------------------------------
VALUE_LABELS = {
    # sex (DHS coding: 1 = male, 2 = female)
    "1": "Male", "2": "Female",
    # residence
    "urban": "Urban", "rural": "Rural",
}

def clean_value(v):
    return VALUE_LABELS.get(str(v).lower(), str(v).replace("_", " ").capitalize())

# --------------------------------------
# ONE FIGURE PER SUBGROUP
# --------------------------------------
n_countries = len(countries)
n_cols = 4
n_rows = int(np.ceil(n_countries / n_cols))

for subgroup in subgroups:

    sdf = df[df["subgroup"] == subgroup].copy()
    sdf["value_label"] = sdf["value"].apply(clean_value)

    # sorted unique values for consistent x-axis order across panels
    value_order = sorted(sdf["value_label"].unique())

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(6 * n_cols, 4 * n_rows),
        sharey=True
    )
    axes = axes.flatten()

    fig.suptitle(
        f"Subgroup: {subgroup}",
        fontsize=14, fontweight="bold", y=1.01
    )

    for i, country in enumerate(countries):

        ax = axes[i]
        cdf = sdf[sdf["country"] == country]

        # mean AUC per (value, model) — averages over CV folds if present
        plot_df = (
            cdf.groupby(["value_label", "model"], as_index=False)["auc"]
            .mean()
        )

        sns.barplot(
            data=plot_df,
            x="value_label",
            y="auc",
            hue="model",
            order=value_order,
            hue_order=list(palette.keys()),
            ax=ax,
            palette=palette,
            edgecolor="black",
            linewidth=0.3
        )

        ax.set_title(clean_country(country), fontweight="bold", pad=6)
        ax.set_ylim(0.5, 1.0)
        ax.set_xlabel("")

        if i % n_cols != 0:
            ax.set_ylabel("")
            ax.set_yticklabels([])
        else:
            ax.set_ylabel("AUC–ROC")

        ax.tick_params(axis="x", rotation=35)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.set_axisbelow(True)
        ax.grid(axis="y", linestyle=":", linewidth=0.4, alpha=0.5)

        if ax.get_legend():
            ax.get_legend().remove()

    # remove unused axes
    for j in range(len(countries), len(axes)):
        fig.delaxes(axes[j])

    # global legend
    handles, labels = axes[0].get_legend_handles_labels()
    legend = fig.legend(
        handles, labels,
        title="Model",
        loc="lower center",
        ncol=4,
        frameon=False,
        fontsize=13,
        title_fontsize=11,
        bbox_to_anchor=(0.5, -0.03)
    )
    legend.get_title().set_fontweight("bold")

    fig.supylabel("AUC–ROC", fontsize=13)
    fig.supxlabel(f"{subgroup}", fontsize=13)

    plt.subplots_adjust(
        left=0.06, right=0.98,
        top=0.94, bottom=0.12,
        hspace=0.60, wspace=0.25
    )

    # save one PDF per subgroup
    safe_name = subgroup.lower().replace(" ", "_")
    plt.savefig(
        f"{FIGURES_DIR}/subgroup_{safe_name}_by_country.pdf",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()
    plt.close()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import numpy as np
import re

# --------------------------------------------------------
# LOAD DATA
# --------------------------------------------------------
# df = pd.read_csv(f"{RESULTS_DIR}/subgroup_analysis_optuna_models.csv")

df = pd.read_csv(f"{RESULTS_DIR}/subgroup_analysis_optuna_models_updated.csv")

# --------------------------------------------------------
# CLEAN LABELS
# --------------------------------------------------------
def clean_country(name):
    name = name.replace("_", " ").replace("DHS", "")
    name = re.sub(r"\s+", " ", name)
    name = re.sub(r"(\d{4})\s+(\d{4})", r"\1–\2", name)
    return name.strip()

def clean_subgroup(name):
    return name.replace("_", " ").capitalize()

VALUE_LABELS = {
    "1": "Male", "2": "Female",
    "urban": "Urban", "rural": "Rural",
}

def clean_value(v):
    return VALUE_LABELS.get(str(v).strip().lower(), str(v).replace("_", " ").capitalize())

df["subgroup"] = df["subgroup"].apply(clean_subgroup)
df["value_label"] = df["value"].apply(clean_value)
df["country"] = df["country"].apply(clean_country)

df["sg_val"] = df["subgroup"] + " | " + df["value_label"]

# --------------------------------------------------------
# AGGREGATE
# --------------------------------------------------------
agg = (
    df.groupby(["country", "sg_val", "model"], as_index=False)["auc"]
    .mean()
)

models = ["LogReg", "LightGBM", "XGBoost", "TabPFN"]
countries = sorted(agg["country"].unique())
sg_vals = sorted(agg["sg_val"].unique())

# x positions (spaced categories)
x_pos = {k: i for i, k in enumerate(sg_vals)}

# --------------------------------------------------------
# STYLE (LANCET READY)
# --------------------------------------------------------
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Liberation Sans", "Helvetica"],
    "font.size": 8,
    "axes.titlesize": 10,
    "axes.labelsize": 8,
    "xtick.labelsize": 6.5,
    "ytick.labelsize": 7,
})

cmap = sns.color_palette("RdYlGn", as_cmap=True)
norm = mcolors.Normalize(vmin=0.50, vmax=1.00)

# --------------------------------------------------------
# FIGURE
# --------------------------------------------------------
fig, axes = plt.subplots(
    2, 2,
    figsize=(10.5, 8.5),
    constrained_layout=False
)
axes = axes.flatten()

# --------------------------------------------------------
# PLOT PER MODEL
# --------------------------------------------------------
for idx, model in enumerate(models):

    ax = axes[idx]

    sub = agg[agg["model"] == model]

    # background light grid lines per subgroup
    for x in range(len(sg_vals)):
        ax.axvline(x, color="#eeeeee", linewidth=0.6, zorder=0)

    # plot each country
    for c in countries:
        cdata = sub[sub["country"] == c]

        x = [x_pos[v] for v in cdata["sg_val"]]
        y = [c] * len(cdata)
        auc = cdata["auc"].values

        # faint connecting line (structure, not emphasis)
        ax.plot(x, y, color="#cccccc", linewidth=0.8, zorder=1)

        # dots (main signal)
        sc = ax.scatter(
            x, y,
            c=auc,
            cmap=cmap,
            norm=norm,
            s=28,
            edgecolor="black",
            linewidth=0.3,
            zorder=2
        )

    ax.set_title(model, fontweight="bold", pad=8)

    ax.set_yticks(countries)
    ax.set_yticklabels(countries)

    ax.set_xticks(range(len(sg_vals)))
    ax.set_xticklabels(sg_vals, rotation=60, ha="right")

    ax.set_xlabel("")
    ax.set_ylabel("")

    # clean spines
    for spine in ax.spines.values():
        spine.set_visible(False)

# --------------------------------------------------------
# SHARED COLORBAR
# --------------------------------------------------------
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

cbar_ax = fig.add_axes([0.91, 0.15, 0.02, 0.70])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label("AUC–ROC", fontsize=8)
cbar.set_ticks([0.50, 0.60, 0.70, 0.80, 0.90, 1.00])
cbar.ax.tick_params(labelsize=7)

# --------------------------------------------------------
# SPACING (IMPORTANT)
# --------------------------------------------------------
plt.subplots_adjust(
    left=0.22,
    right=0.86,   # more room for colorbar + right padding
    top=0.93,
    bottom=0.28,  # more space for rotated x labels
    hspace=0.75,  # ⬅️ vertical spacing between panels (KEY FIX)
    wspace=0.45   # ⬅️ horizontal spacing between panels (KEY FIX)
)

# --------------------------------------------------------
# SAVE (LANCET QUALITY)
# --------------------------------------------------------
plt.savefig(
    f"{FIGURES_DIR}/fig_subgroup_dotline_plot.pdf",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.savefig(
    f"{FIGURES_DIR}/fig_subgroup_dotline_plot.tiff",
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.savefig(
    f"{FIGURES_DIR}/fig_subgroup_dotline_plot.png",
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss
import seaborn as sns
import re
import os

# -----------------------------
# Load data
# -----------------------------
file_path = f"{RESULTS_DIR}/anemia_model_predictions_by_country.csv"
df = pd.read_csv(file_path)

models = ["LogReg", "LightGBM", "XGBoost", "TabPFN"]
countries = sorted(df["country"].dropna().unique())

# -----------------------------
# CLEAN COUNTRY NAMES (FIXED)
# -----------------------------

def clean_country(name):
    name = name.replace("_", " ").replace("DHS", "")
    name = re.sub(r"\s+", " ", name)
    name = re.sub(r"(\d{4})\s+(\d{4})", r"\1–\2", name)
    return name.strip()

countries_clean = [clean_country(c) for c in countries]

# mapping back to cleaned labels
country_map = dict(zip(countries, countries_clean))

df["country_clean"] = df["country"].map(country_map)

# -----------------------------
# STYLE (Lancet-ready)
# -----------------------------
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Liberation Sans"],
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

palette = {
    "LogReg": "#1f77b4",
    "LightGBM": "#ff7f0e",
    "XGBoost": "#2ca02c",
    "TabPFN": "#d62728",
}

# -----------------------------
# FIGURE LAYOUT
# -----------------------------
n_cols = 4
n_rows = int(np.ceil(len(countries) / n_cols))

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(10.5, 3.8 * n_rows),
    sharex=True,
    sharey=True,
    constrained_layout=False
)

axes = axes.flatten()

# -----------------------------
# PLOT PER COUNTRY
# -----------------------------
for i, country in enumerate(countries):

    ax = axes[i]
    sub_country = df[df["country"] == country]

    # perfect calibration line
    ax.plot([0, 1], [0, 1], linestyle="--", color="black", linewidth=1)

    for m in models:
        sub_model = sub_country[sub_country["model"] == m]

        if len(sub_model) < 10:
            continue

        y_true = sub_model["y_true"].values
        y_prob = sub_model["y_prob"].values

        prob_true, prob_pred = calibration_curve(
            y_true,
            y_prob,
            n_bins=10,
            strategy="quantile"
        )

        ax.plot(
            prob_pred,
            prob_true,
            marker="o",
            linewidth=1.6,
            markersize=3.5,
            color=palette[m],
            label=m
        )

    ax.set_title(country_map[country], pad=6)

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    ax.grid(alpha=0.2, linewidth=0.6)

    ax.set_xlabel("")
    ax.set_ylabel("")

    for spine in ax.spines.values():
        spine.set_visible(False)

# -----------------------------
# REMOVE UNUSED AXES
# -----------------------------
for j in range(i + 1, len(axes)):
    axes[j].axis("off")

# -----------------------------
# GLOBAL LEGEND
# -----------------------------
handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=4,
    frameon=False,
    bbox_to_anchor=(0.5, 1.01)
)

# -----------------------------
# GLOBAL LABELS
# -----------------------------
fig.supxlabel("Mean predicted probability", fontsize=9, y=0.04)
fig.supylabel("Observed probability", fontsize=9, x=0.04)

fig.suptitle(
    "Country-stratified calibration",
    fontsize=12,
    y=1.03,
    fontweight="bold"
)

# -----------------------------
# SPACING (CLEAN + BALANCED)
# -----------------------------
plt.subplots_adjust(
    top=0.90,
    bottom=0.10,
    left=0.08,
    right=0.98,
    hspace=0.55,
    wspace=0.25
)

# -----------------------------
# SAVE
# -----------------------------
out_dir = FIGURES_DIR
file_base = "figure_calibration_country_clean"

fig.savefig(
    f"{out_dir}/{file_base}.pdf",
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    f"{out_dir}/{file_base}.tiff",
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    f"{out_dir}/{file_base}.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from dcurves import dca

# -----------------------------
# Load data
# -----------------------------
file_path = f"{RESULTS_DIR}/anemia_model_predictions_by_country.csv"
df = pd.read_csv(file_path)

models = ["LogReg", "LightGBM", "XGBoost", "TabPFN"]
countries = sorted(df["country"].dropna().unique())

# -----------------------------
# CLEAN COUNTRY NAMES
# -----------------------------

def clean_country(name):
    name = name.replace("_", " ").replace("DHS", "")
    name = re.sub(r"\s+", " ", name)
    name = re.sub(r"(\d{4})\s+(\d{4})", r"\1–\2", name)
    return name.strip()

country_map = {c: clean_country(c) for c in countries}
df["country_clean"] = df["country"].map(country_map)

# -----------------------------
# STYLE (Lancet-ready)
# -----------------------------
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Liberation Sans"],
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

palette = {
    "LogReg":   "#1f77b4",
    "LightGBM": "#ff7f0e",
    "XGBoost":  "#2ca02c",
    "TabPFN":   "#d62728",
    "all":      "dimgray",
    "none":     "black",
}

thresholds = np.arange(0.01, 0.99, 0.01)

# -----------------------------
# FIGURE LAYOUT
# -----------------------------
n_cols = 4
n_rows = int(np.ceil(len(countries) / n_cols))

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(10.5, 3.8 * n_rows),
    sharex=True,
    sharey=False,
    constrained_layout=False,
)

axes = axes.flatten()

# -----------------------------
# RUN DCA + PLOT PER COUNTRY
# -----------------------------
for i, country in enumerate(countries):
    ax = axes[i]
    sub = df[df["country"] == country]

    # -- Pivot to wide format: one column per model --
    # dcurves expects outcome + model prob columns in one dataframe
    wide = sub.pivot_table(
        index=sub.groupby("model").cumcount(),   # align rows across models
        columns="model",
        values=["y_true", "y_prob"],
        aggfunc="first",
    )

    # Flatten MultiIndex columns
    wide.columns = [f"{col[0]}_{col[1]}" for col in wide.columns]
    wide = wide.reset_index(drop=True)

    # All models share the same y_true — take from any one
    wide["y_true"] = wide[f"y_true_{models[0]}"]

    # Rename prob columns to model names (what dcurves expects)
    prob_cols = {}
    for m in models:
        col = f"y_prob_{m}"
        if col in wide.columns:
            wide[m] = wide[col]
            prob_cols[m] = m

    available_models = list(prob_cols.keys())

    if len(wide) < 10 or not available_models:
        ax.axis("off")
        continue

    # Drop rows with any NaN in required columns
    cols_needed = ["y_true"] + available_models
    wide = wide[cols_needed].dropna()

    # -- Run dcurves DCA --
    dca_df = dca(
        data=wide,
        outcome="y_true",
        modelnames=available_models,
        thresholds=thresholds,
    )

    # dcurves returns: threshold, model, net_benefit (and net_benefit_all, net_benefit_none)
    # "all" and "none" are included as model rows automatically

    # -- Plot baselines --
    for ref_label, ls in [("none", ":"), ("all", "--")]:
        ref = dca_df[dca_df["model"] == ref_label]
        if ref.empty:
            continue
        ax.plot(
            ref["threshold"],
            ref["net_benefit"],
            linestyle=ls,
            color=palette[ref_label],
            linewidth=1.1,
            label=ref_label.capitalize().replace("None", "Treat none").replace("All", "Treat all"),
        )

    # -- Plot models --
    for m in available_models:
        mdf = dca_df[dca_df["model"] == m]
        if mdf.empty:
            continue
        ax.plot(
            mdf["threshold"],
            np.clip(mdf["net_benefit"], -0.05, None),
            linewidth=1.6,
            color=palette[m],
            label=m,
        )

    ax.set_title(country_map[country], pad=6)
    ax.set_xlim(0, 1)
    ax.set_ylim(bottom=-0.05)
    ax.grid(alpha=0.2, linewidth=0.6)
    ax.set_xlabel("")
    ax.set_ylabel("")

    for spine in ax.spines.values():
        spine.set_visible(False)

# -----------------------------
# REMOVE UNUSED AXES
# -----------------------------
for j in range(i + 1, len(axes)):
    axes[j].axis("off")

# -----------------------------
# GLOBAL LEGEND
# (pull from first populated axis)
# -----------------------------
for ax in axes:
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        break

fig.legend(
    handles, labels,
    loc="upper center",
    ncol=6,
    frameon=False,
    bbox_to_anchor=(0.5, 1.01),
)

# -----------------------------
# GLOBAL LABELS
# -----------------------------
fig.supxlabel("Threshold probability", fontsize=9, y=0.04)
fig.supylabel("Net benefit", fontsize=9, x=0.04)

fig.suptitle(
    "Country-stratified decision curve analysis",
    fontsize=12,
    y=1.03,
    fontweight="bold",
)

# -----------------------------
# SPACING
# -----------------------------
plt.subplots_adjust(
    top=0.90, bottom=0.10,
    left=0.08, right=0.98,
    hspace=0.55, wspace=0.30,
)

# -----------------------------
# SAVE
# -----------------------------
out_dir = FIGURES_DIR
file_base = "figure_dca_country_clean"

fig.savefig(f"{out_dir}/{file_base}.pdf", bbox_inches="tight", facecolor="white")
fig.savefig(f"{out_dir}/{file_base}.tiff", dpi=600, bbox_inches="tight", facecolor="white")
fig.savefig(f"{out_dir}/{file_base}.png",  dpi=300, bbox_inches="tight", facecolor="white")

plt.show()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import brier_score_loss

# -----------------------------
# LOAD DATA
# -----------------------------
file_path = f"{RESULTS_DIR}/anemia_model_predictions_by_country.csv"
df = pd.read_csv(file_path)

models = ["LogReg", "LightGBM", "XGBoost", "TabPFN"]

# -----------------------------
# ECE FUNCTION (quantile-based)
# -----------------------------
def expected_calibration_error(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    bins = np.quantile(y_prob, np.linspace(0, 1, n_bins + 1))
    bins[0] = 0.0
    bins[-1] = 1.0

    bin_ids = np.digitize(y_prob, bins[1:-1], right=True)

    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if mask.sum() == 0:
            continue

        acc = y_true[mask].mean()
        conf = y_prob[mask].mean()
        ece += (mask.sum() / len(y_true)) * abs(acc - conf)

    return ece

# -----------------------------
# COLLECT METRICS
# -----------------------------
rows = []

for m in models:
    sub_model = df[df["model"] == m]

    briers = []
    eces = []

    for country in sub_model["country"].unique():
        sub_c = sub_model[sub_model["country"] == country]

        if len(sub_c) < 10:
            continue

        y_true = sub_c["y_true"].values
        y_prob = sub_c["y_prob"].values

        briers.append(brier_score_loss(y_true, y_prob))
        eces.append(expected_calibration_error(y_true, y_prob))

    rows.append({
        "Model": m,
        "Mean Brier": np.mean(briers),
        "SD Brier": np.std(briers),
        "Mean ECE": np.mean(eces),
        "SD ECE": np.std(eces),
    })

summary_df = pd.DataFrame(rows)

# -----------------------------
# RANKING (lower is better)
# -----------------------------
summary_df["Rank (Brier)"] = summary_df["Mean Brier"].rank(method="min")
summary_df["Rank (ECE)"] = summary_df["Mean ECE"].rank(method="min")

summary_df = summary_df.sort_values("Mean Brier")

# -----------------------------
# ROUNDING FOR PUBLICATION
# -----------------------------
summary_df["Mean Brier"] = summary_df["Mean Brier"].round(4)
summary_df["SD Brier"] = summary_df["SD Brier"].round(4)
summary_df["Mean ECE"] = summary_df["Mean ECE"].round(4)
summary_df["SD ECE"] = summary_df["SD ECE"].round(4)

# -----------------------------
# DISPLAY
# -----------------------------
print("\nAcross-country model calibration performance:\n")
print(summary_df.to_string(index=False))

# -----------------------------
# SAVE TABLES (JOURNAL READY)
# -----------------------------
# summary_df.to_csv(f"{FIGURES_DIR}/calibration_summary_table.csv", index=False)
# summary_df.to_latex(f"{FIGURES_DIR}/calibration_summary_table.tex", index=False)

print(summary_df.to_latex(
    index=False,
    escape=False,
    column_format="lcccc",
    caption="Across-country model calibration performance. Lower values indicate better calibration and predictive reliability.",
    label="tab:calibration_summary"
))

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.metrics import brier_score_loss

# -----------------------------
# LOAD DATA
# -----------------------------
file_path = f"{RESULTS_DIR}/anemia_model_predictions_by_country.csv"
df = pd.read_csv(file_path)

models = ["LogReg", "LightGBM", "XGBoost", "TabPFN"]

# -----------------------------
# CLEAN COUNTRY NAMES
# -----------------------------
def clean_country(name):
    name = name.replace("_", " ").replace("DHS", "")
    name = re.sub(r"\s+", " ", name)
    name = re.sub(r"(\d{4})\s+(\d{4})", r"\1–\2", name)
    return name.strip()

df["country"] = df["country"].apply(clean_country)

# -----------------------------
# ECE FUNCTION (quantile-based)
# -----------------------------
def expected_calibration_error(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    bins = np.quantile(y_prob, np.linspace(0, 1, n_bins + 1))
    bins[0] = 0.0
    bins[-1] = 1.0

    bin_ids = np.digitize(y_prob, bins[1:-1], right=True)

    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if mask.sum() == 0:
            continue

        acc = y_true[mask].mean()
        conf = y_prob[mask].mean()
        ece += (mask.sum() / len(y_true)) * abs(acc - conf)

    return ece

# -----------------------------
# COUNTRY-LEVEL METRICS
# -----------------------------
rows = []

for country in sorted(df["country"].unique()):
    sub_country = df[df["country"] == country]

    for m in models:
        sub = sub_country[sub_country["model"] == m]

        if len(sub) < 10:
            continue

        y_true = sub["y_true"].values
        y_prob = sub["y_prob"].values

        brier = brier_score_loss(y_true, y_prob)
        ece = expected_calibration_error(y_true, y_prob)

        rows.append({
            "Country": country,
            "Model": m,
            "Brier": brier,
            "ECE": ece
        })

country_df = pd.DataFrame(rows)

# -----------------------------
# RANKING WITHIN EACH COUNTRY
# -----------------------------
country_df["Rank (ECE)"] = (
    country_df.groupby("Country")["ECE"]
    .rank(method="min")
)

country_df["Rank (Brier)"] = (
    country_df.groupby("Country")["Brier"]
    .rank(method="min")
)

# -----------------------------
# ROUNDING FOR PUBLICATION
# -----------------------------
country_df["Brier"] = country_df["Brier"].round(4)
country_df["ECE"] = country_df["ECE"].round(4)
country_df["Rank (ECE)"] = country_df["Rank (ECE)"].astype(int)
country_df["Rank (Brier)"] = country_df["Rank (Brier)"].astype(int)

# -----------------------------
# SORT FOR READABILITY
# -----------------------------
country_df = country_df.sort_values(["Country", "Rank (ECE)"])

# -----------------------------
# DISPLAY
# -----------------------------
# print("\nCountry-level calibration performance:\n")
# print(country_df.to_string(index=False))

# -----------------------------
# SAVE TABLES (JOURNAL READY)
# -----------------------------
# CSV
# country_df.to_csv(f"{FIGURES_DIR}/country_calibration_table.csv", index=False)

# LaTeX
print(country_df.to_latex(
    index=False,
    escape=False,
    column_format="llcccc",
    caption="Country-level calibration performance (Brier score and Expected Calibration Error). Rankings are computed within each country (lower is better).",
    label="tab:country_calibration"
))

In [ ]:
"""
make_supplementary_table.py  (wide format)
-------------------------------------------
One row per country. Each subgroup becomes a column whose cell lists
"Label=N" pairs, e.g.  Male=1,204, Female=983
Invalid strata are marked with a dagger.

Prints the LaTeX table to stdout.
"""

import pandas as pd
import numpy as np
from config import RESULTS_DIR

# =========================================================
# LOAD
# =========================================================
df = pd.read_csv(f"{RESULTS_DIR}/supplementary_subgroup_table.csv")

# =========================================================
# LABEL MAPS
# =========================================================
SUBGROUP_ORDER  = ["sex", "wealth", "education", "residence", "age_group"]
SUBGROUP_LABELS = {
    "sex":       "Sex",
    "wealth":    "Wealth",
    "education": "Education",
    "residence": "Residence",
    "age_group": "Age group",
}

VALUE_LABELS = {
    "sex":       {"1": "Male",    "2": "Female"},
    "wealth":    {"1": "Poorest", "2": "Poorer", "3": "Middle", "4": "Richer", "5": "Richest"},
    "education": {"0": "None",    "1": "Primary", "2": "Secondary", "3": "Higher"},
    "residence": {"1": "Urban",   "2": "Rural", "urban": "Urban", "rural": "Rural"},
    "age_group": {},
}

def get_label(subgroup, raw_val):
    lmap = VALUE_LABELS.get(subgroup, {})
    return lmap.get(str(raw_val), str(raw_val))

# =========================================================
# BUILD ONE CELL PER (country, subgroup)
# =========================================================
def build_cell(rows):
    parts = []
    for _, r in rows.iterrows():
        label  = get_label(r["subgroup"], r["value"])
        n      = r["n_after_cleaning"]
        n_fmt  = f"{int(n):,}" if pd.notna(n) and int(n) > 0 else "---"
        dagger = r"$^\dagger$" if not r["ml_valid"] else ""
        parts.append(rf"{label}={n_fmt}{dagger}")
    return ", ".join(parts) if parts else "---"

# =========================================================
# PIVOT
# =========================================================
records = []
for country, cdf in df.groupby("country"):
    row = {"Country": country}
    for sg in SUBGROUP_ORDER:
        sg_rows = cdf[cdf["subgroup"] == sg]
        row[sg] = build_cell(sg_rows) if not sg_rows.empty else r"\textit{n/a}"
    records.append(row)

wide = pd.DataFrame(records).sort_values("Country").reset_index(drop=True)

# =========================================================
# RENDER LATEX BODY
# =========================================================
body_lines = []
for _, r in wide.iterrows():
    cells = [rf"\textbf{{{r['Country']}}}"] + [r[sg] for sg in SUBGROUP_ORDER]
    body_lines.append("  " + " & ".join(cells) + r" \\")
body = "\n".join(body_lines)

COL_W    = "4.0cm"
n_cols   = len(SUBGROUP_ORDER) + 1
col_spec = r"@{}l" + "".join([f"p{{{COL_W}}}" for _ in SUBGROUP_ORDER]) + r"@{}"

col_headers = " & ".join(
    [r"\textbf{Country}"]
    + [rf"\textbf{{{SUBGROUP_LABELS[sg]}}}" for sg in SUBGROUP_ORDER]
)

tex_content = (
    "% =============================================================\n"
    "%  Supplementary Table S1 -- Subgroup sample sizes (wide format)\n"
    "% =============================================================\n\n"
    "\\begin{footnotesize}\n"
    "\\setlength{\\LTleft}{0pt}\n"
    "\\setlength{\\LTright}{0pt}\n\n"
    f"\\begin{{longtable}}{{{col_spec}}}\n\n"
    "  \\caption{%\n"
    "    Complete-case sample sizes per demographic subgroup across all\n"
    "    included DHS surveys. Each cell reports the number of observations\n"
    "    remaining after outcome and covariate filtering for each stratum.\n"
    "    $^\\dagger$~Stratum excluded from ML analyses ($n < 50$ after\n"
    "    filtering, or a single outcome class in the training split).\n"
    "    \\textit{n/a}~indicates the variable was absent from the survey.\n"
    "  }\n"
    "  \\label{tab:subgroup_counts}\\\\\n\n"
    f"  \\toprule\n"
    f"  {col_headers} \\\\\n"
    "  \\midrule\n"
    "  \\endfirsthead\n\n"
    f"  \\multicolumn{{{n_cols}}}{{c}}{{\\tablename\\ \\thetable{{}} -- \\textit{{continued}}}}\\\\[2pt]\n"
    "  \\toprule\n"
    f"  {col_headers} \\\\\n"
    "  \\midrule\n"
    "  \\endhead\n\n"
    "  \\midrule\n"
    f"  \\multicolumn{{{n_cols}}}{{r}}{{\\textit{{Continued on next page}}}}\\\\\n"
    "  \\endfoot\n\n"
    "  \\bottomrule\n"
    "  \\endlastfoot\n\n"
    + body + "\n\n"
    "\\end{longtable}\n"
    "\\end{footnotesize}\n"
)

print(tex_content)